# Pose Alignment 前処理パイプライン（LoRA画像 2点間補間・動画生成向け）

LoRAで生成した2枚の画像（基準画像 `ref`・対象画像 `tgt`）の構図・ポーズ位置を揃えるための前処理ノートブックです。  
動画生成（2点間補間）や後工程の ComfyUI Inpaint を見据え、以下の処理をまとめて実行できます。

1. **セットアップ・データ準備**: 必要パッケージの導入と Drive からの画像コピー
2. **背景色の統一（任意）**: `rembg`（アニメ特化モデル `isnet-anime`）で前景を切り抜き、基準画像の四隅の色に背景を統一
3. **骨格検出（WholeBody）**: `rtmlib` で COCO WholeBody 形式（133点）のキーポイントを抽出
4. **位置合わせ変換**: 剛体変換（スケール1.0固定）または相似変換（スケール調整あり）で行列を算出
5. **欠損チェック & Inpaint用マスク生成**: 変換後の欠損領域（%）と端の箇所を警告し、ComfyUI Inpaint 用白黒マスクを出力

## 1. セットアップ

必要なライブラリ（`rtmlib`, `onnxruntime-gpu`, `opencv-python`, `rembg[gpu]`）をインストールし、GPUランタイムの状態を確認します。

In [ ]:
%pip install -q rtmlib onnxruntime-gpu opencv-python rembg[gpu]

: 

In [1]:
!nvidia-smi

Sun Nov 22 14:18:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Google Drive マウント & データ準備

Google Drive をマウントし、作業用ディレクトリ `./data` を作成してテスト画像をコピーします。  
`mkdir -p ./data` を使用しているため、ディレクトリが既に存在していてもエラーになりません。

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls '/content/drive/MyDrive/Colab Notebooks/sample-img/'
!mkdir -p ./data
!cp '/content/drive/MyDrive/Colab Notebooks/sample-img/000003.png' './data/000003.png'
!cp '/content/drive/MyDrive/Colab Notebooks/sample-img/000004.png' './data/000004.png'
!cp '/content/drive/MyDrive/Colab Notebooks/sample-img/000004-resize.png' './data/000004-resize.png'

ls: cannot access '/content/drive/MyDrive/Colab Notebooks/img-sample/': No such file or directory
cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/img-sample/000003.png': No such file or directory
cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/img-sample/000004.png': No such file or directory
cp: cannot stat '/content/drive/MyDrive/Colab Notebooks/img-sample/000004-resize.png': No such file or directory


## 3. ユーティリティ関数・モデル初期化

再利用しやすいように、モデルのセッション初期化と各処理のユーティリティ関数を定義します。

- `get_bg_color` / `unify_background`: 背景色統一（`isnet-anime` セッションを使い回し）
- `estimate_rigid_transform`: SVDを用いた剛体変換（スケール `1.0` 固定・鏡像反転補正あり）
- `warp_with_check`: `cv2.warpAffine` による変換と、欠損領域の検知・Inpaint用マスク生成
- `show_comparison` / `show_mask_overlay`: `matplotlib` を用いた枠線付き比較表示・マスク重畳表示
- `save_results_to_drive`: 生成画像・マスクを Google Drive へ保存

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from rtmlib import Wholebody, draw_skeleton
from rembg import remove, new_session

# --- モデルセッションの初期化（初回のみONNXモデルが自動DLされます） ---
# 1. 骨格検出モデル（GPUランタイムなら device='cuda'、CPUなら 'cpu'）
wholebody = Wholebody(
    to_openpose=False,  # COCO WholeBody形式（133点）を使用
    mode='performance', # 高精度なRTMW系モデルを使用
    backend='onnxruntime',
    device='cuda'
)

# 2. 背景切り抜き用アニメ特化モデル（セッションを使い回して高速化）
anime_session = new_session("isnet-anime")

In [ ]:
def get_bg_color(img):
    """
    画像の四隅のピクセルから背景色（BGRの中央値）を自動取得する。

    Parameters
    ----------
    img : np.ndarray
        入力画像（BGR形式）。

    Returns
    -------
    np.ndarray
        四隅のBGR中央値（uint8, shape=(3,)）。
    """
    corners = np.array([
        img[0, 0], img[0, -1],
        img[-1, 0], img[-1, -1]
    ])
    return np.median(corners, axis=0).astype(np.uint8)


def unify_background(tgt_img, ref_img=None, bg_color_bgr=None, session=anime_session):
    """
    対象画像（tgt_img）の背景を切り抜き、基準画像（ref_img）の四隅の色または指定BGR色で統一する。
    ※ warpAffine の前に実行しておくと、画像端が引き伸ばされる（BORDER_REPLICATE）際に
       境界のにじみが抑えられ、綺麗な単色背景として補完しやすくなります。

    Parameters
    ----------
    tgt_img : np.ndarray
        背景を統一したい対象画像（BGR形式）。
    ref_img : np.ndarray, optional
        背景色の参照元となる基準画像（BGR形式）。bg_color_bgr 未指定時に使用。
    bg_color_bgr : tuple or list or np.ndarray, optional
        直接指定する背景色（BGR形式）。指定した場合は ref_img より優先される。
    session : rembg.sessions.BaseSession, optional
        使い回す rembg セッション（デフォルトはアニメ特化モデル isnet-anime）。

    Returns
    -------
    np.ndarray
        背景色統一後の画像（BGR形式, uint8）。
    """
    # 1. 背景色の決定（指定色 -> 基準画像の四隅中央値 -> 白 の優先順）
    if bg_color_bgr is None:
        bg_color_bgr = get_bg_color(ref_img) if ref_img is not None else np.array([255, 255, 255], dtype=np.uint8)
    else:
        bg_color_bgr = np.array(bg_color_bgr, dtype=np.uint8)

    # 2. アニメ特化モデルで前景（キャラクター）をRGBAで切り抜き
    tgt_rgb = cv2.cvtColor(tgt_img, cv2.COLOR_BGR2RGB)
    fg_rgba = remove(tgt_rgb, session=session)

    # 3. アルファチャンネルを使って新しい背景色と綺麗に合成
    alpha = fg_rgba[:, :, 3:4].astype(np.float32) / 255.0
    fg_bgr = cv2.cvtColor(fg_rgba[:, :, :3], cv2.COLOR_RGB2BGR).astype(np.float32)
    bg_canvas = np.full_like(fg_bgr, bg_color_bgr, dtype=np.float32)

    result = (fg_bgr * alpha + bg_canvas * (1.0 - alpha)).astype(np.uint8)
    return result

In [ ]:
def estimate_rigid_transform(src_pts, dst_pts):
    """
    SVD（特異値分解）を用いて、拡大縮小なし（スケール=1.0固定）のまま
    位置（平行移動）と回転だけを最適に合わせる2x3変換行列（剛体変換）を計算する。
    ※ キャラクターの頭身や LoRA 特有の線幅・奥行き感を崩さず維持したい場合に使用します。

    Parameters
    ----------
    src_pts : np.ndarray
        変換元のキーポイント座標（shape=(N, 2)）。
    dst_pts : np.ndarray
        変換先（基準画像）のキーポイント座標（shape=(N, 2)）。

    Returns
    -------
    np.ndarray
        cv2.warpAffine にそのまま渡せる 2x3 の変換行列（float32）。
    """
    # 1. 各点群の重心（中心座標）を求める
    src_mean = np.mean(src_pts, axis=0)
    dst_mean = np.mean(dst_pts, axis=0)

    # 2. 重心を原点に揃える
    src_centered = src_pts - src_mean
    dst_centered = dst_pts - dst_mean

    # 3. SVD（特異値分解）で最適な回転行列 R だけを求める
    H = src_centered.T @ dst_centered
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # 鏡像反転（行列式が負＝裏返し）が起きてしまった場合の補正
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    # 4. 平行移動ベクトル t を計算
    t = dst_mean - R @ src_mean

    # 5. OpenCVの warpAffine で使える 2x3 行列にまとめる
    M = np.hstack([R, t.reshape(2, 1)])
    return M.astype(np.float32)


def warp_with_check(tgt_img, M, out_size, warn_threshold_pct=0.5):
    """
    cv2.warpAffine（INTER_LANCZOS4, BORDER_REPLICATE）で画像を変換し、
    元画像でカバーできない欠損領域の割合（%）と発生箇所（上下左右の端）を検知して警告を出す。
    また、欠損領域を白（255）にした ComfyUI Inpaint 用のマスク画像を返す。

    Parameters
    ----------
    tgt_img : np.ndarray
        変換対象の画像（BGR形式）。
    M : np.ndarray
        2x3 のアフィン変換行列。
    out_size : tuple[int, int]
        出力画像サイズ (width, height)。
    warn_threshold_pct : float, optional
        警告ログを出す欠損率のしきい値（%）。デフォルトは 0.5%。

    Returns
    -------
    aligned_img : np.ndarray
        位置合わせ後の画像（BGR形式）。
    missing_mask : np.ndarray
        欠損領域が白（255）、有効領域が黒（0）の1chマスク画像（ComfyUI Inpaint用）。
    """
    w, h = out_size

    # 1. 本体の変換（高品質補間 LANCZOS4 + 端は周辺色で自然に見せる BORDER_REPLICATE）
    aligned_img = cv2.warpAffine(
        tgt_img, M, (w, h),
        flags=cv2.INTER_LANCZOS4,
        borderMode=cv2.BORDER_REPLICATE
    )

    # 2. 元画像と同じサイズの白マスクを同じ行列で変換（外側は 0=黒 で埋める）
    valid_mask = np.full(tgt_img.shape[:2], 255, dtype=np.uint8)
    warped_valid = cv2.warpAffine(
        valid_mask, M, (w, h),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    # 欠損領域（0の部分）を抽出し、Inpaint用マスクとして 白=255 に反転
    missing_mask = (warped_valid == 0).astype(np.uint8) * 255
    missing_pixels = np.count_nonzero(missing_mask)
    missing_pct = (missing_pixels / (w * h)) * 100

    # 3. 欠損がしきい値を超えていたら発生箇所とともに警告を出す
    if missing_pct >= warn_threshold_pct:
        edges = []
        if np.any(missing_mask[0, :] > 0):  edges.append("上端")
        if np.any(missing_mask[-1, :] > 0): edges.append("下端")
        if np.any(missing_mask[:, 0] > 0):  edges.append("左端")
        if np.any(missing_mask[:, -1] > 0): edges.append("右端")

        print(f"⚠️ 警告: 元画像で補完できない領域が {missing_pct:.2f}% 発生しています（箇所: {', '.join(edges)}）")
    else:
        print(f"✅ OK: 欠損なし（{missing_pct:.2f}%）")

    return aligned_img, missing_mask

In [ ]:
import os
import shutil

def _style_image_axis(ax, title, border_color='gray'):
    """画像表示用の軸から目盛りを消し、画像の境界が分かるように枠線を表示する。"""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_color)
        spine.set_linewidth(1.5)


def show_comparison(img_left, img_right, title_left='Reference', title_right='Aligned Target', figsize=(10, 5)):
    """
    Colab上で安全に確認できるよう、matplotlibを用いて2枚の画像を横に並べて比較表示する。
    カラー画像（BGR）は自動でRGB変換し、1chマスク画像はグレースケールとして表示する。
    白・黒の背景と同化しないよう、各画像の周囲に枠線を描画する。

    Parameters
    ----------
    img_left : np.ndarray
        左側に表示する画像（基準画像など）。
    img_right : np.ndarray
        右側に表示する画像（変換後画像やマスクなど）。
    title_left : str, optional
        左画像のタイトル。
    title_right : str, optional
        右画像のタイトル。
    figsize : tuple[int, int], optional
        matplotlib の図サイズ。
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    for ax, img, title in zip(axes, [img_left, img_right], [title_left, title_right]):
        if img.ndim == 2:
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        _style_image_axis(ax, title)

    plt.tight_layout()
    plt.show()


def show_mask_overlay(aligned_img, missing_mask, figsize=(15, 5)):
    """
    位置合わせ後画像・Inpaint用マスク（枠線付き）・赤色オーバーレイの3枚を並べて表示する。
    マスクの白黒領域が背景と同化して境界が分かりにくくなるのを防ぎ、
    画像内のどこに欠損が発生しているかを直感的に確認できる。

    Parameters
    ----------
    aligned_img : np.ndarray
        位置合わせ後の画像（BGR形式）。
    missing_mask : np.ndarray
        欠損領域が白（255）、有効領域が黒（0）の1chマスク画像。
    figsize : tuple[int, int], optional
        matplotlib の図サイズ。
    """
    # 元画像に欠損領域を半透明の赤で重ねたプレビューを作成（保存用画像には影響しない）
    overlay = aligned_img.copy()
    mask_bool = missing_mask > 0
    overlay[mask_bool] = (
        overlay[mask_bool].astype(np.float32) * 0.45
        + np.array([0, 0, 255], dtype=np.float32) * 0.55
    ).astype(np.uint8)

    # 欠損領域の輪郭線も描画して境界を強調
    contours, _ = cv2.findContours(missing_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(overlay, contours, -1, (0, 255, 255), 2)

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    axes[0].imshow(cv2.cvtColor(aligned_img, cv2.COLOR_BGR2RGB))
    _style_image_axis(axes[0], 'Aligned Target')

    axes[1].imshow(missing_mask, cmap='gray', vmin=0, vmax=255)
    _style_image_axis(axes[1], 'Inpaint Mask (White = Missing)', border_color='red')

    axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    _style_image_axis(axes[2], 'Mask Overlay (Red = Missing, Cyan = Border)')

    plt.tight_layout()
    plt.show()


def save_results_to_drive(output_dir, file_paths=None, prefix=''):
    """
    生成した画像やマスクファイルを Google Drive 上の指定ディレクトリへ保存する。

    Parameters
    ----------
    output_dir : str
        保存先の Google Drive ディレクトリパス。
    file_paths : list[str], optional
        保存対象ファイルパスのリスト。未指定時は標準の出力ファイルを対象とする。
    prefix : str, optional
        保存ファイル名の先頭に付与するプレフィックス（例: 'cut01_'）。
    """
    if file_paths is None:
        file_paths = [
            'aligned_target.png',
            'inpaint_mask.png',
            'unified_target.png',
            'debug_ref.png',
            'debug_tgt.png',
        ]

    if not os.path.exists('/content/drive'):
        print("⚠️ 警告: '/content/drive' が見つかりません。先に Drive をマウントしてください。")
        return

    os.makedirs(output_dir, exist_ok=True)
    saved_count = 0

    for src_path in file_paths:
        if not os.path.exists(src_path):
            print(f"ℹ️ スキップ（未生成）: {src_path}")
            continue
        dst_name = f"{prefix}{os.path.basename(src_path)}"
        dst_path = os.path.join(output_dir, dst_name)
        shutil.copy2(src_path, dst_path)
        print(f"✅ 保存完了: {dst_path}")
        saved_count += 1

    print(f"合計 {saved_count} ファイルを Drive に保存しました -> {output_dir}")

## 4. パラメータ設定＆位置合わせ実行

セル上部のフラグ・パラメータを切り替えることで、用途に応じた位置合わせを簡単に試せます。

### 変換モード（`USE_FIXED_SCALE`）の選び方
- `True`（**剛体変換**）: スケールを `1.0` に固定し、平行移動と回転のみを行います。**LoRAの線幅や奥行き感を維持したい場合**に適しています。
- `False`（**相似変換** `cv2.estimateAffinePartial2D`）: 拡大・縮小も含めて位置合わせを行います。**広い画角の画像を狭い画角にマッチさせたい場合**（例: `000004-resize.png` などのサイズ違い）に適しています。

### 骨格キーポイント（`USE_INDICES`）による演出の違い
COCO WholeBody 形式の主なインデックス：`0:鼻, 1:左目, 2:右目, 5:左肩, 6:右肩, 11:左腰, 12:右腰`
- `[1, 2, 5, 6]`（**両目＋両肩**）: 顔を強く固定します。**表情変化やカメラ追従っぽく見せたいとき**向け（バストアップで安定）。
- `[5, 6, 11, 12]`（**両肩＋両腰**）: 体幹を基準にします。**歩行や重心移動の自然な揺れを残したいとき**向け（全身・膝上で安定）。

In [ ]:
# ==========================================
# パラメータ設定（ここを切り替えて実行）
# ==========================================
REF_PATH = './data/000004.png'          # 基準にしたい構図の画像
TGT_PATH = './data/000003.png'          # 位置合わせしたい対象画像
# TGT_PATH = './data/000004-resize.png' # 画角・サイズ違いの実験用パス

UNIFY_BG = True                         # True: 変換前に背景色を統一（端の引き伸ばしが綺麗な単色になる）
BG_COLOR_BGR = None                     # None: 基準画像の四隅中央値を自動取得 / 例: (220, 220, 220)
USE_FIXED_SCALE = True                  # True: 剛体変換（スケール固定）/ False: 相似変換（スケール変更あり）
USE_INDICES = [1, 2, 5, 6]              # [1, 2, 5, 6]: 両目+両肩（顔固定）/ [5, 6, 11, 12]: 両肩+両腰（体幹基準）
WARN_THRESHOLD_PCT = 0.5                # 欠損警告を出すしきい値（%）

# ==========================================
# 1. 画像読み込み
# ==========================================
ref_img = cv2.imread(REF_PATH)
tgt_img = cv2.imread(TGT_PATH)
assert ref_img is not None, f"基準画像が読み込めません: {REF_PATH}"
assert tgt_img is not None, f"対象画像が読み込めません: {TGT_PATH}"

# ==========================================
# 2. 背景色統一（warpAffineの前に実行）
# ==========================================
if UNIFY_BG:
    # 変換前に背景を統一しておくと、BORDER_REPLICATE による端の引き伸ばしが綺麗な単色になります
    proc_tgt_img = unify_background(tgt_img, ref_img=ref_img, bg_color_bgr=BG_COLOR_BGR)
    cv2.imwrite('unified_target.png', proc_tgt_img)
else:
    proc_tgt_img = tgt_img

# ==========================================
# 3. 骨格キーポイント検出（WholeBody）
# ==========================================
ref_kpts, ref_scores = wholebody(ref_img)
tgt_kpts, tgt_scores = wholebody(proc_tgt_img)

# 1人目のキャラクターの座標を取得
ref_p = ref_kpts[0]
tgt_p = tgt_kpts[0]

src_pts = np.array([tgt_p[i] for i in USE_INDICES], dtype=np.float32)
dst_pts = np.array([ref_p[i] for i in USE_INDICES], dtype=np.float32)

# ==========================================
# 4. 変換行列の計算
# ==========================================
if USE_FIXED_SCALE:
    M = estimate_rigid_transform(src_pts, dst_pts)
    print("変換モード: 剛体変換（スケール=1.0固定）")
else:
    M, _ = cv2.estimateAffinePartial2D(src_pts, dst_pts)
    print("変換モード: 相似変換（スケール調整あり）")

# ==========================================
# 5. 変換実行 & 欠損チェック
# ==========================================
h, w = ref_img.shape[:2]
aligned_img, missing_mask = warp_with_check(
    proc_tgt_img, M, (w, h), warn_threshold_pct=WARN_THRESHOLD_PCT
)

# ==========================================
# 6. 結果とデバッグ画像の保存
# ==========================================
cv2.imwrite('aligned_target.png', aligned_img)
cv2.imwrite('inpaint_mask.png', missing_mask)

debug_ref = draw_skeleton(ref_img.copy(), ref_kpts, ref_scores, kpt_thr=0.5)
debug_tgt = draw_skeleton(proc_tgt_img.copy(), tgt_kpts, tgt_scores, kpt_thr=0.5)
cv2.imwrite('debug_ref.png', debug_ref)
cv2.imwrite('debug_tgt.png', debug_tgt)

# ==========================================
# 7. 基準画像と変換後画像の比較表示
# ==========================================
show_comparison(ref_img, aligned_img, title_left='Reference', title_right='Aligned Target')

In [ ]:
# 欠損マスク（ComfyUI Inpaint用：枠線・オーバーレイ付き）と骨格検出デバッグ画像の確認
show_mask_overlay(aligned_img, missing_mask)
show_comparison(debug_ref, debug_tgt, title_left='Debug Skeleton: Reference', title_right='Debug Skeleton: Target')

## 5. 結果のDrive保存・ランタイム終了メモ

生成した `aligned_target.png` や ComfyUI Inpaint 用の `inpaint_mask.png` などを Google Drive へ保存します。  
複数カットを連続で処理する場合は `OUTPUT_PREFIX` を設定すると上書きを防げます。

In [ ]:
# ==========================================
# Google Drive への結果保存
# ==========================================
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/img/output'
OUTPUT_PREFIX = ''  # 例: 'scene01_' を付けると上書きを防止できます

# 保存したいファイル（不要なものはコメントアウト可能）
TARGET_FILES = [
    'aligned_target.png',
    'inpaint_mask.png',
    'unified_target.png',
    'debug_ref.png',
    'debug_tgt.png',
]

save_results_to_drive(DRIVE_OUTPUT_DIR, file_paths=TARGET_FILES, prefix=OUTPUT_PREFIX)

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する